# 🎬 ClipGod — worker GPU (Kaggle)

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/moonkamel/clipgod/blob/main/notebooks/clipgod_kaggle.ipynb)

Alternative à Colab (≈ 30 h de GPU gratuites par semaine). Même rôle : traiter les jobs de **Google Drive ›
`ClipGod/jobs/`** avec **scan** puis **render**, et écrire les clips dans `ClipGod/clips/`.

**Réglages, une seule fois** (panneau de droite) :
1. *Session options › Accelerator* : **GPU T4 x2** ; *Internet* : **On** (compte Kaggle vérifié par téléphone).
2. *Add-ons › Secrets* (cochez-les pour ce notebook) :
   - `CLIPGOD_GDRIVE_TOKEN_JSON` = contenu du fichier `secrets/drive_token.json` du laptop
     (créé au premier `clipgod prepare` / `clipgod sync`) ;
   - `GITHUB_TOKEN` (optionnel, seulement si le repo est privé).

Puis **Run All**. Pour une longue VOD sans garder l'onglet ouvert : *Save Version › Save & Run All*
(exécution en arrière-plan, jusqu'à 12 h). Ensuite, sur le laptop : `clipgod sync`.

> Kaggle n'a pas de Drive monté : tout passe par l'API Drive (écritures immédiates, rien à synchroniser).
> Si la session s'arrête après la transcription, la relance réutilise `ClipGod/moments/<id>.transcript.json`.

In [ ]:
# ⚙️ Paramètres
REPO_URL = "https://github.com/moonkamel/clipgod.git"
BRANCH = "main"
STAGES = "scan+render"          # "scan+render" | "scan" | "render"
VOICEOVER = False               # voix off Piper
MAX_JOBS = 0                    # 0 = tous
ONLY_JOB = ""                   # vide = file normale
WHISPER_MODEL = "auto"          # auto | large-v3 | medium | small
REPO_DIR = "/kaggle/working/clipgod"
WORK_DIR = "/tmp/clipgod_work"  # hors /kaggle/working (limité à 20 Go et archivé avec les outputs)

In [ ]:
# 1️⃣ Vérification du GPU
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, check=False)
if gpu.returncode != 0:
    raise RuntimeError("❌ Aucun GPU. Session options › Accelerator › GPU T4 x2, puis Run All.")
print("✅ GPU :", gpu.stdout.strip())

In [ ]:
# 2️⃣ Secrets Kaggle -> accès Google Drive par API
import os
import urllib.request

from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()


def _secret(name):
    try:
        return _secrets.get_secret(name) or ""
    except Exception:  # noqa: BLE001 — secret absent ou non attaché à ce notebook
        return ""


token_json = _secret("CLIPGOD_GDRIVE_TOKEN_JSON")
if not token_json:
    raise RuntimeError("❌ Secret CLIPGOD_GDRIVE_TOKEN_JSON manquant (Add-ons › Secrets) : collez le contenu de "
                       "secrets/drive_token.json du laptop, cochez-le pour ce notebook, puis Run All.")
os.environ["CLIPGOD_GDRIVE_TOKEN_JSON"] = token_json
GITHUB_TOKEN = _secret("GITHUB_TOKEN")
try:
    urllib.request.urlopen("https://www.googleapis.com/discovery/v1/apis", timeout=15)
except OSError as exc:
    raise RuntimeError("❌ Pas d'accès Internet : Session options › Internet › On.") from exc
print("✅ Token Drive chargé, Internet OK")

In [ ]:
# 3️⃣ Récupération du code ClipGod
import os
import subprocess


def _git(*args, cwd=None):
    proc = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if proc.returncode != 0:
        err = proc.stderr.strip()
        if token:
            err = err.replace(token, "***")        # ne jamais afficher le token
        raise RuntimeError(f"git {args[0]} a échoué : {err}")
    return proc.stdout.strip()

token = GITHUB_TOKEN
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _git("fetch", "--depth", "1", auth_url, BRANCH, cwd=REPO_DIR)
    _git("reset", "--hard", "FETCH_HEAD", cwd=REPO_DIR)
else:
    _git("clone", "--depth", "1", "--branch", BRANCH, auth_url, REPO_DIR)
# Le clone enregistre l'URL (avec le token) dans .git/config : on la remplace par l'URL publique.
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)
print("✅ ClipGod", BRANCH, "@", _git("rev-parse", "--short", "HEAD", cwd=REPO_DIR))

In [ ]:
# 4️⃣ Installation des dépendances (≈ 2 min)
!bash {REPO_DIR}/scripts/colab_setup.sh {"--with-tts" if VOICEOVER else ""}

In [ ]:
# 5️⃣ Initialisation
import os
import sys

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if "core.worker" in sys.modules:
    print("ℹ️ Code déjà importé : après une mise à jour du repo, faites Run › Restart & clear cell outputs.")

from core import worker
from core.config import load_config
from core.drive import get_drive
from core.logger import setup_logging

cfg = load_config(os.path.join(REPO_DIR, "config.yaml"))
setup_logging(cfg, force=True)
drive = get_drive(cfg)          # Kaggle -> API Drive (token du secret)
drive.ensure_structure()
print("✅ Prêt :", drive)

In [ ]:
# 6️⃣ Jobs en attente
import pandas as pd

rows = worker.queue_status(drive)
if not rows:
    print("📭 Aucun job en attente. Sur le laptop : clipgod prepare --source <url>")
else:
    display(pd.DataFrame(rows))

In [ ]:
# 7️⃣ Traitement (scan puis render)
stages = {"scan": ("scan",), "scan+render": ("scan", "render"), "render": ("render",)}[STAGES]
overrides = {}
if WHISPER_MODEL != "auto":
    overrides["scan"] = {"whisper": {"model": WHISPER_MODEL}}
if VOICEOVER:
    overrides["render"] = {"voiceover": {"enabled": True}}

results = worker.run_queue(
    drive, cfg,
    workdir=WORK_DIR,
    stages=stages,
    max_jobs=MAX_JOBS or None,
    only_job=ONLY_JOB or None,
    overrides=overrides,
)
if results:
    cols = [c for c in ("job_id", "status", "video_id", "n_moments", "best_score", "clips", "duration_s", "error")
            if any(c in r for r in results)]
    display(pd.DataFrame(results)[cols])
else:
    print("📭 Rien à traiter.")

In [ ]:
# 8️⃣ Aperçu des moments et du premier clip
from base64 import b64encode
from pathlib import Path

from IPython.display import HTML

for r in results:
    if r.get("status") != "done" or not r.get("moments"):
        continue
    doc = drive.read_json(r["moments"])
    print(f"\n🎬 {doc['video_id']} — scorer {doc['scorer']['name']} — signaux : {', '.join(doc['signals_available'])}")
    display(pd.DataFrame([
        {"#": m["rank"], "début": m["start"], "fin": m["end"], "score": m["score"],
         "raison": m["reason"], "hooks": " | ".join(m["hooks"])}
        for m in doc["moments"]
    ]))
    clips = r.get("clips") or []
    print(f"🎞️ {len(clips)} clip(s) dans ClipGod/clips/ :", ", ".join(c.split("/")[-1] for c in clips))
    local = Path(WORK_DIR) / r["job_id"] / "clips" / clips[0].split("/")[-1] if clips else None
    if local is not None and local.is_file():   # copie locale du rendu : pas de relecture depuis Drive
        data = b64encode(local.read_bytes()).decode()
        display(HTML(f'<video width=270 controls src="data:video/mp4;base64,{data}"></video>'))

In [ ]:
# 9️⃣ Terminé
print("✅ Résultats sur Google Drive (ClipGod/clips/). Sur le laptop : clipgod sync")